
# 03 - AcmeNet Basic Retrieval

Goal:
Search relevant support knowledge chunks based on a user question.

Input:
acmenet_silver_chunks

Output:
Top matching chunks with sources and scores.

In [0]:
silver_df = spark.table("acmenet_silver_chunks")

display(
    silver_df.select(
        "chunk_id",
        "document_name",
        "section",
        "source",
        "chunk_text",
        "chunk_word_count"
    )
)

chunk_id,document_name,section,source,chunk_text,chunk_word_count
troubleshooting_guide_001,troubleshooting_guide.md,Slow Internet,troubleshooting,"Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support.",30
billing_policy_001,billing_policy.md,Billing Disputes,billing,Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.,25
refund_policy_001,refund_policy.md,Refund Eligibility,refunds,Customers may request a refund within 30 days of activation if the service was unavailable for more than 72 continuous hours due to a provider-side issue.,26


In [0]:
chunk_count = silver_df.count()

print(f"Knowledge chunks available: {chunk_count}")

if chunk_count == 0:
      raise ValueError("No chunks found. Please run Notebook 02 first.")

Knowledge chunks available: 3


In [0]:
# Define question
question = "My internet is slow. What should I do?"

print(question)

My internet is slow. What should I do?


In [0]:
# Extract keywords
import re

STOPWORDS = {
    "the", "is", "are", "a", "an", "and", "or", "to", "of", "in",
    "my", "i", "what", "should", "do", "can", "if", "it", "for",
    "with", "on", "this", "that", "be"
}

def extract_keywords(question: str) -> list[str]:
    # Normalize
    normalized = question.lower()
    normalized = re.sub(r"[^a-z0-9\s]", " ", normalized)
    # Convert to words
    words = normalized.split()

    keywords = [
        word for word in words
        if word not in STOPWORDS and len(words) >= 3
    ]

    return keywords

keywords = extract_keywords(question)

print(keywords)

['internet', 'slow']


In [0]:
# Matching function

def find_matched_keywords(text: str) -> list[str]:
    if text is None:
        return []
    
    text_lower = text.lower()

    matched = [
        keyword for keyword in keywords
        if keyword in text_lower
    ]
    return matched
    

In [0]:
# Convert function into User Defined Function
from pyspark.sql.functions import udf, col, size, desc
from pyspark.sql.types import ArrayType, StringType

find_matched_keywords_udf = udf(
    find_matched_keywords,
    ArrayType(StringType())
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:103: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


In [0]:
# Get chunk scoring

scored_df = (
    silver_df
    .withColumn("matched_keywords", find_matched_keywords_udf(col("chunk_text")))
    .withColumn("score", size(col("matched_keywords")))
)

display(
    scored_df.select(
        "chunk_id",
        "document_name",
        "section",
        "matched_keywords",
        "score",
        "chunk_text"
    ).orderBy(desc("score"))
)

chunk_id,document_name,section,matched_keywords,score,chunk_text
billing_policy_001,billing_policy.md,Billing Disputes,"List(dispute, invoice)",2,Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.
troubleshooting_guide_001,troubleshooting_guide.md,Slow Internet,List(),0,"Customers experiencing slow internet should restart the modem by unplugging it for 30 seconds. If speed remains below 50% of the contracted plan after restart, escalate to Tier 2 support."
refund_policy_001,refund_policy.md,Refund Eligibility,List(),0,Customers may request a refund within 30 days of activation if the service was unavailable for more than 72 continuous hours due to a provider-side issue.


In [0]:
# Get top chunks
top_k = 3

retrieved_df = (
    scored_df
    .filter(col("score") > 0)
    .orderBy(desc("score"))
    .limit(top_k)
)

display(
    retrieved_df.select(
        "document_name",
        "section",
        "score",
        "matched_keywords",
        "chunk_text"
    )
)

document_name,section,score,matched_keywords,chunk_text
billing_policy.md,Billing Disputes,2,"List(dispute, invoice)",Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.


In [0]:
# Build context

retrieved_rows = retrieved_df.collect()

context_blocks = []

for row in retrieved_rows:
    context_blocks.append(
        f"Source: {row['document_name']}\n"
        f"Section: {row['section']}\n"
        f"Content: {row['chunk_text']}"
    )

context = "\n\n---\n\n".join(context_blocks)

print(context)

Source: billing_policy.md
Section: Billing Disputes
Content: Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.


In [0]:
# Create simple grounded answer

if not retrieved_rows:
    answer = "I can only answer questions related to AcmeNet support policies, troubleshooting, billing, refunds, installation, and escalation rules."
    sources = []
    status = "out_of_domain"
else:
    answer = (
        "Based on AcmeNet support documentation, here is the most relevant information:\n\n"
        + "\n\n".join([row["chunk_text"] for row in retrieved_rows])
    )
    sources = sorted(list(set([row["document_name"] for row in retrieved_rows])))
    status = "answered"

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

print("\nSources:")
print(sources)

print("\nStatus:")
print(status)

Question:
Can I dispute a charge on my invoice?

Answer:
Based on AcmeNet support documentation, here is the most relevant information:

Customers can request a billing clarification within 15 days of receiving the invoice. Billing disputes above 100 dollars must be escalated to Tier 2 support.

Sources:
['billing_policy.md']

Status:
answered


In [0]:
# Test billing question
# After running this cell run from cell 6 to 11

question = "Can I dispute a charge on my invoice?"
keywords = extract_keywords(question)

print("Question:", question)
print("Keywords:", keywords)

Question: Can I dispute a charge on my invoice?
Keywords: ['dispute', 'charge', 'invoice']


In [0]:
# Test refund question
# After running this cell run from cell 6 to 11

question = "Can I get a refund if my service was down for more than 72 hours?"
keywords = extract_keywords(question)

print("Question:", question)
print("Keywords:", keywords)

In [0]:
# Test question outside domains
# After running this cell run from cell 6 to 11

question = "Can you recommend a gaming laptop?"
keywords = extract_keywords(question)

print("Question:", question)
print("Keywords:", keywords)